# Day 12 — cross-paper retrieval (no LLM calls)

QASPER questions always come with their paper. Cross-paper questions don't ("Which papers run experiments on
HotpotQA, and what does each propose?"), so retrieval must first find the right **papers** across all 1,585.

Today we measure that, before any LLM is involved:
1. 19 hand-written questions (`eval/cross_paper_questions.jsonl`), each with the papers it should find. The set was
   fixed before this code existed.
2. Four ways to search the whole corpus: BM25, dense (FAISS), hybrid (RRF), hybrid + cross-encoder reranker.
3. Score: **paper recall@10** = share of the required papers among the first 10 distinct papers found.

Step 1 runs fine on CPU. Step 2 (the reranker) needs a GPU.

In [ ]:
# WHAT: Mounts Google Drive and makes the research-engine folder the working directory.
# WHY:  All commands below run from the project root.
from google.colab import drive
drive.mount('/content/drive')

import os, glob
PROJECT = glob.glob('/content/drive/MyDrive/research-engine')[0]
os.chdir(PROJECT)
print(os.getcwd())

Mounted at /content/drive
/content/drive/MyDrive/research-engine


In [ ]:
# WHAT: Installs the project's packages (run again after every runtime restart).
# WHY:  A fresh Colab session doesn't have rank_bm25, sentence-transformers or faiss.
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 11.3 MB/s eta 0:00:00


## Step 1 — BM25, dense and hybrid (CPU is fine)

In [ ]:
# WHAT: Searches the whole corpus for each of the 19 questions with BM25, dense (FAISS) and hybrid, groups the
#       ranked chunks by paper, and scores paper recall@5/10/20, precision@10 and MRR.
# WHY:  Shows which search finds the right papers best, without the slow reranker. Takes 1-2 minutes.
#       Saved as results/cross_paper_retrieval_norerank.json/.md.
!python -m eval.run_cross_paper_eval --no-rerank --tag norerank

modules.json: 100% 349/349 [00:00<00:00, 1.27MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 660kB/s]
README.md: 100% 94.8k/94.8k [00:00<00:00, 108MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 286kB/s]
config.json: 100% 743/743 [00:00<00:00, 4.01MB/s]

model.safetensors: downloading bytes:  57% 75.4M/133M [00:01<00:00, 101MB/s, 5.99MB/s  ] 
model.safetensors: reconstructing file:  85% 113M/133M [00:01<00:00, 94.2MB/s]
model.safetensors: downloading bytes: 100% 86.7M/86.7M [00:01<00:00, 60.1MB/s, 8.09MB/s  ]
model.safetensors: reconstructing file: 100% 133M/133M [00:01<00:00, 92.5MB/s, 12.7MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 6174.98it/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 1.36MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 41.6MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 80.0MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 653kB/s]
config.json: 100% 190/190 [00:00<00:00, 853kB/s]
Loading cached embeddings: /

## Step 2 — add the cross-encoder reranker (GPU only)
Only run this on a **T4 GPU** runtime (Runtime → Change runtime type). On CPU the reranker is far too slow.

In [ ]:
# WHAT: Same as step 1, plus bge-reranker-v2-m3 re-scoring the top 100 fused chunks of each question (1,900 pairs).
# WHY:  Within one paper the reranker added +3.8 points R@5 (Day 4); this checks whether it also helps to find papers.
#       Scores are cached in cache/, so a rerun is instant. Saved as results/cross_paper_retrieval.json/.md.
import torch
assert torch.cuda.is_available(), "No GPU: switch to a T4 runtime first (Runtime -> Change runtime type)."
!python -m eval.run_cross_paper_eval

Loading weights: 100% 199/199 [00:00<00:00, 4829.21it/s]
Loading cached embeddings: /content/drive/MyDrive/research-engine/cache/emb_BAAI_bge-small-en-v1.5_28541.npy
19 cross-paper questions | 28541 chunks | dense backend=flat
FAISS IndexFlatIP built in 0.03 s (28541 vectors, dim 384)
config.json: 100% 795/795 [00:00<00:00, 3.63MB/s]

model.safetensors: downloading bytes:  15% 341M/2.27G [00:01<00:10, 185MB/s, 29.7MB/s  ]
model.safetensors: downloading bytes:  16% 373M/2.27G [00:02<00:10, 176MB/s, 32.4MB/s  ]
model.safetensors: downloading bytes:  22% 490M/2.27G [00:02<00:08, 202MB/s, 41.9MB/s  ]
model.safetensors: downloading bytes:  37% 830M/2.27G [00:04<00:11, 127MB/s, 62.7MB/s  ]
model.safetensors: downloading bytes:  39% 897M/2.27G [00:05<00:08, 160MB/s, 65.8MB/s  ]
model.safetensors: downloading bytes:  42% 955M/2.27G [00:05<00:08, 150MB/s, 69.2MB/s  ]
model.safetensors: downloading bytes:  44% 999M/2.27G [00:05<00:08, 150MB/s, 70.8MB/s  ]
model.safetensors: downloading bytes:  5

In [ ]:
# WHAT: Shows the saved result tables.
# WHY:  To read the numbers and paste them into the results log.
from IPython.display import Markdown, display
for f in ["results/cross_paper_retrieval_norerank.md", "results/cross_paper_retrieval.md"]:
    if os.path.exists(f):
        display(Markdown(open(f).read()))

# Cross-paper retrieval (19 questions, paper-level)

| system | paper_recall@5 | paper_recall@10 | paper_recall@20 | paper_prec@10 | mrr |
|---|---|---|---|---|---|
| bm25 | 0.487 | 0.713 | 0.860 | 0.679 | 0.882 |
| bm25 [dataset] | 0.596 | 0.761 | 0.857 | 0.600 | 0.950 |
| bm25 [topic] | 0.366 | 0.661 | 0.864 | 0.767 | 0.806 |
| dense | 0.479 | 0.725 | 0.851 | 0.674 | 0.888 |
| dense [dataset] | 0.449 | 0.674 | 0.786 | 0.540 | 0.787 |
| dense [topic] | 0.513 | 0.782 | 0.924 | 0.822 | 1.000 |
| hybrid | 0.499 | 0.735 | 0.906 | 0.705 | 0.895 |
| hybrid [dataset] | 0.511 | 0.711 | 0.873 | 0.590 | 0.867 |
| hybrid [topic] | 0.486 | 0.763 | 0.942 | 0.833 | 0.926 |

## Per question (paper recall@10)

| id | type | required | bm25 R@10 | dense R@10 | hybrid R@10 |
|---|---|---|---|---|---|
| XP01 | dataset | 6 | 0.67 | 1.00 | 1.00 |
| XP02 | dataset | 8 | 0.75 | 0.75 | 0.62 |
| XP03 | dataset | 8 | 0.50 | 0.88 | 0.75 |
| XP04 | dataset | 7 | 0.86 | 0.57 | 0.86 |
| XP05 | dataset | 8 | 0.62 | 0.38 | 0.50 |
| XP06 | dataset | 6 | 0.83 | 0.00 | 0.33 |
| XP07 | dataset | 3 | 1.00 | 0.67 | 0.67 |
| XP08 | dataset | 3 | 1.00 | 1.00 | 1.00 |
| XP09 | dataset | 8 | 0.75 | 0.62 | 0.75 |
| XP10 | dataset | 8 | 0.62 | 0.88 | 0.62 |
| XP11 | topic | 11 | 0.55 | 0.82 | 0.73 |
| XP12 | topic | 10 | 0.60 | 0.60 | 0.60 |
| XP13 | topic | 12 | 0.50 | 0.75 | 0.58 |
| XP14 | topic | 7 | 0.71 | 0.86 | 0.86 |
| XP15 | topic | 7 | 0.57 | 0.71 | 0.71 |
| XP16 | topic | 12 | 0.58 | 0.75 | 0.75 |
| XP17 | topic | 12 | 0.83 | 0.75 | 0.83 |
| XP18 | topic | 4 | 1.00 | 1.00 | 1.00 |
| XP19 | topic | 5 | 0.60 | 0.80 | 0.80 |


# Cross-paper retrieval (19 questions, paper-level)

| system | paper_recall@5 | paper_recall@10 | paper_recall@20 | paper_prec@10 | mrr |
|---|---|---|---|---|---|
| bm25 | 0.487 | 0.713 | 0.860 | 0.679 | 0.882 |
| bm25 [dataset] | 0.596 | 0.761 | 0.857 | 0.600 | 0.950 |
| bm25 [topic] | 0.366 | 0.661 | 0.864 | 0.767 | 0.806 |
| dense | 0.479 | 0.725 | 0.851 | 0.674 | 0.888 |
| dense [dataset] | 0.449 | 0.674 | 0.786 | 0.540 | 0.787 |
| dense [topic] | 0.513 | 0.782 | 0.924 | 0.822 | 1.000 |
| hybrid | 0.499 | 0.735 | 0.906 | 0.705 | 0.895 |
| hybrid [dataset] | 0.511 | 0.711 | 0.873 | 0.590 | 0.867 |
| hybrid [topic] | 0.486 | 0.763 | 0.942 | 0.833 | 0.926 |
| hybrid+rerank | 0.556 | 0.840 | 0.959 | 0.758 | 0.974 |
| hybrid+rerank [dataset] | 0.624 | 0.886 | 0.975 | 0.690 | 0.950 |
| hybrid+rerank [topic] | 0.481 | 0.788 | 0.942 | 0.833 | 1.000 |

## Per question (paper recall@10)

| id | type | required | bm25 R@10 | dense R@10 | hybrid R@10 | hybrid+rerank R@10 |
|---|---|---|---|---|---|---|
| XP01 | dataset | 6 | 0.67 | 1.00 | 1.00 | 1.00 |
| XP02 | dataset | 8 | 0.75 | 0.75 | 0.62 | 0.75 |
| XP03 | dataset | 8 | 0.50 | 0.88 | 0.75 | 0.88 |
| XP04 | dataset | 7 | 0.86 | 0.57 | 0.86 | 0.86 |
| XP05 | dataset | 8 | 0.62 | 0.38 | 0.50 | 0.75 |
| XP06 | dataset | 6 | 0.83 | 0.00 | 0.33 | 1.00 |
| XP07 | dataset | 3 | 1.00 | 0.67 | 0.67 | 1.00 |
| XP08 | dataset | 3 | 1.00 | 1.00 | 1.00 | 1.00 |
| XP09 | dataset | 8 | 0.75 | 0.62 | 0.75 | 1.00 |
| XP10 | dataset | 8 | 0.62 | 0.88 | 0.62 | 0.62 |
| XP11 | topic | 11 | 0.55 | 0.82 | 0.73 | 0.82 |
| XP12 | topic | 10 | 0.60 | 0.60 | 0.60 | 0.80 |
| XP13 | topic | 12 | 0.50 | 0.75 | 0.58 | 0.67 |
| XP14 | topic | 7 | 0.71 | 0.86 | 0.86 | 0.86 |
| XP15 | topic | 7 | 0.57 | 0.71 | 0.71 | 0.57 |
| XP16 | topic | 12 | 0.58 | 0.75 | 0.75 | 0.75 |
| XP17 | topic | 12 | 0.83 | 0.75 | 0.83 | 0.83 |
| XP18 | topic | 4 | 1.00 | 1.00 | 1.00 | 1.00 |
| XP19 | topic | 5 | 0.60 | 0.80 | 0.80 | 0.80 |
